# Open the prepared SPY chart dataset
Loads the completed 2020–2025 research dataset from your saved Colab notebook. Choose an example timestamp to view the two charts or save the six annual PNG archives to your Google Drive. The default chart shows a completed five-minute entry on November 19, 2024. This notebook uses the saved dataset without a new market-data download.

In [ ]:
# @title Load and verify the prepared dataset
source = 'My Drive' # @param ['My Drive', 'Upload saved notebook']
from pathlib import Path
import base64, hashlib, io, json, sys, zipfile
import pandas as pd
from google.colab import drive, files
if source == 'My Drive':
    drive.mount('/content/drive')
    notebook = Path('/content/drive/MyDrive/Colab Notebooks/SPY_Paired_Daily_5min_2020_2025.ipynb')
    if not notebook.exists():
        raise FileNotFoundError('Choose Upload saved notebook if you moved or renamed the Drive copy.')
    book = json.loads(notebook.read_text())
else:
    uploads = files.upload()
    book = json.loads(next(iter(uploads.values())))
parts = []
for cell in book['cells']:
    for output in cell.get('outputs', []):
        part = output.get('data',{}).get('application/json',{})
        if isinstance(part,dict) and part.get('format') == 'SPY-pair-data-v1':
            parts.append(part)
parts.sort(key=lambda p:p['part'])
assert [p['part'] for p in parts] == list(range(58)), 'Select the 27.9 MB private notebook with the embedded dataset.'
raw = b''.join(base64.b64decode(p['payload'],validate=True) for p in parts)
assert hashlib.sha256(raw).hexdigest() == 'f5d5493c34d4d55dc734cfd6349bc9e72680caee906a1083442116b54ad18209'
out = Path('/content/prepared_spy')
out.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    assert z.testzip() is None
    z.extractall(out)
(out/'SPY_Paired_2020_2025_Data.zip').write_bytes(raw)
sys.path.insert(0,str(out))
import paired_decision_charts as charts
daily,bars,pairs = charts.load_views(out)
print(f'{len(pairs):,} decisions across {pairs.index.normalize().nunique():,} sessions.')
print(pairs.groupby('split').size())
setups = pd.read_parquet(out/'call_setup_outcomes.parquet')
display(setups[setups.first_eligible_signal_of_session].groupby(['split','outcome']).size())

In [ ]:
# @title View a chart pair
decision = '2024-11-19 15:40' # @param {type:"string"}
display(charts.rich_pair(daily,bars,pairs,decision))

In [ ]:
# @title Export the dataset or a year of prepared PNG charts
selection = 'Dataset and outcomes' # @param ['Dataset and outcomes','2020','2021','2022','2023','2024','2025','All years']
save_to = 'Browser download' # @param ['Browser download','My Drive']
import shutil
dest = out
if save_to == 'My Drive':
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    dest = Path('/content/drive/MyDrive/SPY_Paired_Charts')
    dest.mkdir(exist_ok=True)
if selection == 'Dataset and outcomes':
    outputs = [out/'SPY_Paired_2020_2025_Data.zip']
else:
    years = range(2020,2026) if selection == 'All years' else [int(selection)]
    outputs = [charts.year_images(out,year,daily,bars,pairs) for year in years]
if save_to == 'My Drive':
    for asset in outputs:
        tmp = dest/(asset.name+'.partial')
        shutil.copyfile(asset,tmp)
        if hashlib.sha256(tmp.read_bytes()).digest() != hashlib.sha256(asset.read_bytes()).digest():
            raise IOError('Archive copy failed integrity verification')
        tmp.replace(dest/asset.name)
        print(dest/asset.name)
elif len(outputs) == 1:
    files.download(str(outputs[0]))
else:
    combined = out/'SPY_PNG_2020_2025.zip'
    with zipfile.ZipFile(combined,'w',allowZip64=True) as z:
        for asset in outputs: z.write(asset,asset.name)
    files.download(str(combined))